In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').getOrCreate()

In [40]:
# Import all functions
from pyspark.sql.functions import *
from pyspark.sql.types import *

### `from_json(col, schema)`
- Convert a JSON string into a MapType or ArrayType or StructType
- Returns NULL in case of an unparseable string

Map

In [44]:
data = [(1, '{"ZipCode": 85016, "ZipCodeType": "STANDARD", "City": "Phoenix", "State": "AZ"}'),]
df = spark.createDataFrame(data, ['id', 'value'])

# Before
df.show(truncate=False)
df.printSchema()

# After
map_schema = MapType(StringType(), StringType())
df_map = df.withColumn('value', from_json('value', map_schema))
df_map.show(truncate=False)
df_map.printSchema()

+---+-------------------------------------------------------------------------------+
|id |value                                                                          |
+---+-------------------------------------------------------------------------------+
|1  |{"ZipCode": 85016, "ZipCodeType": "STANDARD", "City": "Phoenix", "State": "AZ"}|
+---+-------------------------------------------------------------------------------+

root
 |-- id: long (nullable = true)
 |-- value: string (nullable = true)

+---+-------------------------------------------------------------------------+
|id |value                                                                    |
+---+-------------------------------------------------------------------------+
|1  |{ZipCode -> 85016, ZipCodeType -> STANDARD, City -> Phoenix, State -> AZ}|
+---+-------------------------------------------------------------------------+

root
 |-- id: long (nullable = true)
 |-- value: map (nullable = true)
 |    |-- key: string


Array

In [46]:
data = [(1, "[1, 2, 3]")]
df = spark.createDataFrame(data, ['id', 'value'])

# Before
df.show(truncate=False)
df.printSchema()

# After
arr_schema = ArrayType(IntegerType())
df_arr = df.withColumn('value', from_json('value', arr_schema))
df_arr.show(truncate=False)
df_arr.printSchema()

+---+---------+
|id |value    |
+---+---------+
|1  |[1, 2, 3]|
+---+---------+

root
 |-- id: long (nullable = true)
 |-- value: string (nullable = true)

+---+---------+
|id |value    |
+---+---------+
|1  |[1, 2, 3]|
+---+---------+

root
 |-- id: long (nullable = true)
 |-- value: array (nullable = true)
 |    |-- element: integer (containsNull = true)



Struct

In [49]:
data = [(1, '{"ZipCode": 85016, "ZipCodeType": "STANDARD", "City": "Phoenix", "State": "AZ"}'),]
df = spark.createDataFrame(data, ['id', 'value'])

# Before
df.show(truncate=False)
df.printSchema()

# After
struct_schema = StructType([StructField("ZipCode", IntegerType()), StructField("ZipCodeType", StringType()), StructField("City", StringType()), StructField("State", StringType())])
df_struct = df.withColumn('value', from_json('value', struct_schema))
df_struct.show(truncate=False)
df_struct.printSchema()

+---+-------------------------------------------------------------------------------+
|id |value                                                                          |
+---+-------------------------------------------------------------------------------+
|1  |{"ZipCode": 85016, "ZipCodeType": "STANDARD", "City": "Phoenix", "State": "AZ"}|
+---+-------------------------------------------------------------------------------+

root
 |-- id: long (nullable = true)
 |-- value: string (nullable = true)

+---+------------------------------+
|id |value                         |
+---+------------------------------+
|1  |{85016, STANDARD, Phoenix, AZ}|
+---+------------------------------+

root
 |-- id: long (nullable = true)
 |-- value: struct (nullable = true)
 |    |-- ZipCode: integer (nullable = true)
 |    |-- ZipCodeType: string (nullable = true)
 |    |-- City: string (nullable = true)
 |    |-- State: string (nullable = true)



### `to_json(col)`
- Converts a column with StructType or ArrayType or MapType into a JSON string
- Throws an exception in case of an unsupported type

Map

In [51]:
df_map_to_json = df_map.withColumn('value', to_json('value'))
df_map_to_json.show(truncate=False)
df_map_to_json.printSchema()

+---+--------------------------------------------------------------------------+
|id |value                                                                     |
+---+--------------------------------------------------------------------------+
|1  |{"ZipCode":"85016","ZipCodeType":"STANDARD","City":"Phoenix","State":"AZ"}|
+---+--------------------------------------------------------------------------+

root
 |-- id: long (nullable = true)
 |-- value: string (nullable = true)



Array

In [52]:
df_arr_to_json = df_arr.withColumn('value', to_json('value'))
df_arr_to_json.show(truncate=False)
df_arr_to_json.printSchema()

+---+-------+
|id |value  |
+---+-------+
|1  |[1,2,3]|
+---+-------+

root
 |-- id: long (nullable = true)
 |-- value: string (nullable = true)



Struct

In [53]:
df_struct_to_json = df_struct.withColumn('value', to_json('value'))
df_struct_to_json.show(truncate=False)
df_struct_to_json.printSchema()

+---+------------------------------------------------------------------------+
|id |value                                                                   |
+---+------------------------------------------------------------------------+
|1  |{"ZipCode":85016,"ZipCodeType":"STANDARD","City":"Phoenix","State":"AZ"}|
+---+------------------------------------------------------------------------+

root
 |-- id: long (nullable = true)
 |-- value: string (nullable = true)



### `json_tuple(col, fields)`
- Creates a new row for a JSON column according to the given field names.

In [54]:
df_map_to_json.select(json_tuple('value', 'ZipCode', 'City')).show()

+-----+-------+
|   c0|     c1|
+-----+-------+
|85016|Phoenix|
+-----+-------+



In [56]:
# To name the columns
df_map_to_json.select(json_tuple('value', 'ZipCode', 'City')).toDF('Zip', 'City').show()

+-----+-------+
|  Zip|   City|
+-----+-------+
|85016|Phoenix|
+-----+-------+



### `schema_of_json(col)`
- Parses a JSON string and infers its schema in STRUCT format

In [65]:
# Sample JSON string
sample_json = '{"name": "John", "age": 30, "city": "New York", "skills": ["Python", "Spark"]}'

# Get schema from JSON string
json_schema = schema_of_json(lit(sample_json))
spark.range(1).select(json_schema).show(truncate=False)

+----------------------------------------------------------------------------------------------+
|schema_of_json({"name": "John", "age": 30, "city": "New York", "skills": ["Python", "Spark"]})|
+----------------------------------------------------------------------------------------------+
|STRUCT<age: BIGINT, city: STRING, name: STRING, skills: ARRAY<STRING>>                        |
+----------------------------------------------------------------------------------------------+



### `get_json_object(col, path)`
- Extracts JSON object from a JSON string based on the JSON path(JSON key) passed
- It will return NULL if the input json string is invalid.

In [73]:
data = [("1", '{"f1": "value1", "f2": "value2"}'), ("2", '{"f1": "value3"}')]
df = spark.createDataFrame(data, ("key", "jstring"))

# Before
df.show(truncate=False)

# After
df.select('jstring', get_json_object('jstring', '$.f1').alias('f1'), get_json_object('jstring', '$.f2').alias('f2') ).show(truncate=False)

+---+--------------------------------+
|key|jstring                         |
+---+--------------------------------+
|1  |{"f1": "value1", "f2": "value2"}|
|2  |{"f1": "value3"}                |
+---+--------------------------------+

+--------------------------------+------+------+
|jstring                         |f1    |f2    |
+--------------------------------+------+------+
|{"f1": "value1", "f2": "value2"}|value1|value2|
|{"f1": "value3"}                |value3|NULL  |
+--------------------------------+------+------+

